In [1]:
import numpy as np
import torch

import gymnasium as gym

from som_rl.som.som_backend import SharpZSOMBackend, SharpZBackendConfig
from som_rl.env.schedules import SchedulerConfig, make_scheduler
from som_rl.env.env import SOMEnergyEnvConfig
from som_rl.env.env_multi import EnvBandit, EnvMinesweeper, EnvZelda
from som_rl.policies.models import BanditMLPPolicy, MinesweeperCNNPolicy, ZeldaCNNPolicy

print("torch version:", torch.__version__)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("RL device:", device)

torch version: 2.10.0
RL device: mps


In [2]:
# Load SOM / correlation data (same paths as 09-RL_Test)
DATA = "data"  # adjust if needed

npz = np.load(f"{DATA}/correlations_dl/npairs_compact.npz")
dd, dr, rd, rr = npz["dd"], npz["dr"], npz["rd"], npz["rr"]

node_indices_photo = np.load(f"{DATA}/node_indices_fc.npy")  # (Nphot,)
initial_node_bins  = np.load(f"{DATA}/nodes_labels_initial_fc_200_ext.npy")  # (40000,)
reference_labels   = np.load(f"{DATA}/reference_labels_fc_fc.npy")  # (Nref,) in [0..10]

weighted_clusters  = np.load(f"{DATA}/weighted_clusters_5c_1t200.npy")  # (L,40000)
umatrix = np.load(f"{DATA}/umat.npy")  # (200,200)

b_cfg = SharpZBackendConfig(
    grid_shape=(200, 200),
    num_bins=11,
    objective_mode="energy",
    proposal_bin_mode="uniform_other",
)


def make_backend():
    return SharpZSOMBackend(
        cfg=b_cfg,
        dd=dd, dr=dr, rd=rd, rr=rr,
        node_indices_photo=node_indices_photo,
        initial_node_bins=initial_node_bins,
        reference_labels=reference_labels,
        umatrix=umatrix,
        weighted_clusters=weighted_clusters,
    )


som = make_backend()
print("Initial energy_E():", som.energy_E())

Initial energy_E(): 0.023775093565945846


In [3]:
# Build a level-hits scheduler via make_scheduler
levels = (30, 50, 80)
HITS_PER_LEVEL = 200

sched_cfg = SchedulerConfig(
    max_steps=2000,
    level_min=min(levels),
    level_max=max(levels),
    level_list=levels,
    min_accepts_per_level=HITS_PER_LEVEL,
    reset_temp_on_level_up=True,
)

scheduler = make_scheduler("level_hits", sched_cfg)
print("Scheduler initial params (level, T, floor):", scheduler.current_params())
print("Scheduler done():", scheduler.done())

Scheduler initial params (level, T, floor): (30, 1.0, 1e-05)
Scheduler done(): False


In [4]:
# Common env config backbone
base_env_cfg = SOMEnergyEnvConfig(
    grid_shape=(200, 200),
    obs_mode="global",          # will be overridden in EnvZelda
    patch_size=9,
    include_umatrix=True,
    include_cluster_features=True,
    include_empty_mask=True,
    include_visit_count=True,
    include_recent_mask=True,
    recent_cooldown_steps=10,
    max_steps=2000,
    initial_level=levels[0],
    initial_temperature=1.0,
    reward_type="deltaE",
    use_tanh_reward=True,
    reward_tanh_temperature=1e-2,
    reward_energy_scale=1.0,
    clip_reward=1.0,
    reject_penalty=0.0,
    noop_penalty=0.0,
    use_greedy_bin_for_node=False,
)

# Fresh backend per env to avoid shared state
som_bandit = make_backend()
som_mine   = make_backend()
som_zelda  = make_backend()

sched_bandit = make_scheduler("level_hits", sched_cfg)
sched_mine   = make_scheduler("level_hits", sched_cfg)
sched_zelda  = make_scheduler("level_hits", sched_cfg)

env_bandit = EnvBandit(som_bandit, base_env_cfg, scheduler=sched_bandit)
env_mine   = EnvMinesweeper(som_mine,   base_env_cfg, scheduler=sched_mine)
env_zelda  = EnvZelda(som_zelda,        base_env_cfg, scheduler=sched_zelda)

for name, env in [("Bandit", env_bandit), ("Minesweeper", env_mine), ("Zelda", env_zelda)]:
    obs, info = env.reset()
    print(f"{name} obs keys:", list(obs.keys()))
    print(f"{name} action space:", env.action_space)
    print("  scalars shape:", obs["scalars"].shape, "cursor shape:", obs["cursor"].shape)
    print("---")

Bandit obs keys: ['scalars', 'cursor']
Bandit action space: Discrete(40000)
  scalars shape: (4,) cursor shape: (2,)
---
Minesweeper obs keys: ['bins', 'cursor', 'empty', 'umatrix', 'cluster_boundary', 'cluster_size', 'visit_count', 'recent', 'scalars']
Minesweeper action space: Discrete(40000)
  scalars shape: (4,) cursor shape: (2,)
---
Zelda obs keys: ['bins', 'cursor', 'empty', 'umatrix', 'cluster_boundary', 'cluster_size', 'visit_count', 'recent', 'scalars']
Zelda action space: Discrete(5)
  scalars shape: (4,) cursor shape: (2,)
---


In [5]:
# Quick rollout sanity checks for each env

def run_random_rollout(env, n_steps: int = 10, name: str = "env"):
    obs, info = env.reset()
    total_reward = 0.0
    for t in range(n_steps):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if (t + 1) % 5 == 0 or terminated or truncated:
            print(f"{name} step {t+1}: r={reward:.4f}, term={terminated}, trunc={truncated}")
        if terminated or truncated:
            break
    print(f"{name} total reward over {t+1} steps: {total_reward:.4f}\n")

run_random_rollout(env_bandit, n_steps=15, name="Bandit")
run_random_rollout(env_mine,   n_steps=15, name="Minesweeper")
run_random_rollout(env_zelda,  n_steps=15, name="Zelda")

Bandit step 5: r=-0.0122, term=False, trunc=False
Bandit step 10: r=-0.0136, term=False, trunc=False
Bandit step 15: r=-0.0216, term=False, trunc=False
Bandit total reward over 15 steps: -0.2093

Minesweeper step 5: r=-0.0133, term=False, trunc=False
Minesweeper step 10: r=-0.0195, term=False, trunc=False
Minesweeper step 15: r=-0.0074, term=False, trunc=False
Minesweeper total reward over 15 steps: -0.3382

Zelda step 5: r=0.0000, term=False, trunc=False
Zelda step 10: r=0.0000, term=False, trunc=False
Zelda step 15: r=0.0000, term=False, trunc=False
Zelda total reward over 15 steps: -0.0018



In [6]:
# Policy shape checks: make sure models accept env observations

# --- Bandit policy ---
obs_b, _ = env_bandit.reset()
bandit_policy = BanditMLPPolicy(action_dim=env_bandit.action_space.n)

bandit_obs_t = {
    "scalars": torch.from_numpy(obs_b["scalars"][None, :]).float(),
    "cursor":  torch.from_numpy(obs_b["cursor"][None, :]).float(),
}
logits_b, value_b = bandit_policy(bandit_obs_t)
print("Bandit logits shape:", logits_b.shape, "value shape:", value_b.shape)

# --- Minesweeper policy ---
obs_m, _ = env_mine.reset()

# Stack available grid-like channels into a single (C,H,W) tensor
grid_keys = [k for k in ["bins", "empty", "umatrix", "cluster_boundary", "cluster_size", "visit_count", "recent"] if k in obs_m]
grid_arrays = [obs_m[k].astype(np.float32) for k in grid_keys]
C = len(grid_arrays)
H, W = grid_arrays[0].shape

grid = np.stack(grid_arrays, axis=0)  # (C,H,W)

mine_policy = MinesweeperCNNPolicy(action_dim=env_mine.action_space.n, grid_channels=C)

mine_obs_t = {
    "grid":    torch.from_numpy(grid[None, ...]),
    "scalars": torch.from_numpy(obs_m["scalars"][None, :]).float(),
    "cursor":  torch.from_numpy(obs_m["cursor"][None, :]).float(),
}
logits_m, value_m = mine_policy(mine_obs_t)
print("Minesweeper logits shape:", logits_m.shape, "value shape:", value_m.shape)

# --- Zelda policy ---
obs_z, _ = env_zelda.reset()

patch_keys = [k for k in ["bins", "empty", "umatrix", "cluster_boundary", "cluster_size", "visit_count", "recent"] if k in obs_z]
patch_arrays = [obs_z[k].astype(np.float32) for k in patch_keys]
Cz = len(patch_arrays)
ps_h, ps_w = patch_arrays[0].shape

patch = np.stack(patch_arrays, axis=0)  # (C,ps,ps)

zelda_policy = ZeldaCNNPolicy(action_dim=env_zelda.action_space.n, patch_channels=Cz)

zelda_obs_t = {
    "patch":   torch.from_numpy(patch[None, ...]),
    "scalars": torch.from_numpy(obs_z["scalars"][None, :]).float(),
    "cursor":  torch.from_numpy(obs_z["cursor"][None, :]).float(),
}
logits_z, value_z = zelda_policy(zelda_obs_t)
print("Zelda logits shape:", logits_z.shape, "value shape:", value_z.shape)

Bandit logits shape: torch.Size([1, 40000]) value shape: torch.Size([1, 1])
Minesweeper logits shape: torch.Size([1, 40000]) value shape: torch.Size([1, 1])
Zelda logits shape: torch.Size([1, 5]) value shape: torch.Size([1, 1])
